In [ ]:
# %%
# Environment & Repository Cloning
!git clone https://github.com/your-username/company-cad-proposal.git
%cd company-cad-proposal
!pip install -q torch torchvision opencv-python python-doctr matplotlib pydantic
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
print('Environment setup complete')


In [ ]:
# %%
# Mount Google Drive and locate drawing.pdf
from google.colab import drive
drive.mount('/content/drive')
import os
pdf_path = "/content/drive/MyDrive/company cad project/drawing.pdf"
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found at {pdf_path}")
print(f"Found drawing.pdf at: {pdf_path}")
input_pdf = pdf_path


In [ ]:
# %%
# Execute Nodes 01 to 04 (Isolated Pre-LLM)
import sys
sys.path.insert(0, 'src')

# Import required nodes
from nodes.triage import PixelTriageNode
from nodes.vectorize import GeometricExtractionNode
from nodes.layout import LayoutExtractionNode
from nodes.dhmot import DHMoTNode

# Initialize nodes
triage_node = PixelTriageNode(node_id='node_01_triage')
vectorize_node = GeometricExtractionNode(node_id='node_02_vectorize')
layout_node = LayoutExtractionNode(node_id='node_03_layout')
dhmot_node = DHMoTNode(node_id='node_04_dhmot')

# Execute pipeline up to Node 04
# Node 01: Pixel Triage
print('Running Node 01: Pixel Triage...')
triage_output = triage_node.execute(input_pdf)
if not triage_output.success:
    print('Triage failed:', triage_output.errors)
else:
    print('Triage successful')
    geometry_mask_path = triage_output.data.geometry_mask_path
    text_mask_path = triage_output.data.text_mask_path
    table_mask_path = triage_output.data.table_mask_path

# Node 02: Geometric Extraction
print('Running Node 02: Geometric Extraction...')
vectorize_output = vectorize_node.execute(geometry_mask_path, page_number=1)
if not vectorize_output.success:
    print('Vectorize failed:', vectorize_output.errors)
else:
    print('Vectorize successful')
    geometry = vectorize_output.data

# Node 03: Layout Extraction
print('Running Node 03: Layout Extraction...')
layout_output = layout_node.execute(table_mask_path, text_mask_path, page_number=1)
if not layout_output.success:
    print('Layout failed:', layout_output.errors)
else:
    print('Layout successful')
    tables = layout_output.data

# Node 04: DHMoT Agent
print('Running Node 04: DHMoT Agent...')
dhmot_output = dhmot_node.execute(
    geometry,
    tables,
    original_img_path=geometry_mask_path
)
if not dhmot_output.success:
    print('DHMoT failed:', dhmot_output.errors)
else:
    print('DHMoT successful')
    axiom_manifest = dhmot_output.data  # This is the Axiom Manifest
    validation_overlay_path = dhmot_output.metadata.get('validation_overlay_path')


In [ ]:
# %%
# Display Deterministic Outputs and Download Files
import json
from IPython.display import Image, display
from google.colab import files
import os

# Print Axiom Manifest as formatted JSON
print('\n=== AXIOM MANIFEST (Pre-LLM Output) ===')\nprint(json.dumps(axiom_manifest, indent=4, default=str))

# Save Axiom Manifest to file for download
manifest_json = json.dumps(axiom_manifest, indent=4, default=str)
with open('axiom_manifest.json', 'w') as f:
    f.write(manifest_json)

# Display validation overlay if available
if validation_overlay_path and os.path.exists(validation_overlay_path):
    print('\n=== VALIDATION OVERLAY ===')
    try:
        display(Image(filename=validation_overlay_path))
    except Exception as e:
        print(f'Could not load overlay image: {e}')
else:
    print('\nValidation overlay path not found or file does not exist')

# Download files
print('\n=== DOWNLOADING OUTPUTS ===')
try:
    files.download('axiom_manifest.json')
    print('Downloaded: axiom_manifest.json')
except Exception as e:
    print(f'Error downloading axiom_manifest.json: {e}')

if validation_overlay_path and os.path.exists(validation_overlay_path):
    try:
        files.download(validation_overlay_path)
        print(f'Downloaded: {os.path.basename(validation_overlay_path)}')
    except Exception as e:
        print(f'Error downloading validation overlay: {e}')
else:
    print('Validation overlay not available for download.')

print('\nIsolated pre-LLM test completed successfully. Outputs are ready for download.')
